# UMI in MuJoCo - Phase 4: Convert the demos to UMI's dataset format

UMI trains from one zarr file: a *replay buffer* that holds every frame of every demo back to back, plus the list of where each episode ends. This notebook turns your Phase 3 demos into that exact format, so UMI's own training code reads them without modification.

**What goes into the file** (read from UMI's `07_generate_replay_buffer.py` and `umi_dataset.py`):

| key | shape per frame | stored as |
|---|---|---|
| `camera0_rgb` | 224 x 224 x 3 | uint8, one JPEG-XL-compressed chunk per frame (level 99, as UMI does) |
| `robot0_eef_pos` | 3 | float32 |
| `robot0_eef_rot_axis_angle` | 3 | float32 |
| `robot0_gripper_width` | 1 | float32 |
| `robot0_demo_start_pose` / `robot0_demo_end_pose` | 6 | float64, the first / last pose of the episode repeated on every frame |
| `meta/episode_ends` | one per episode | int64 |

The start pose is **required**: the dataset class builds the observation `eef_rot_axis_angle_wrt_start` from it. There is no separate action array. Training builds the actions on the fly from the future poses and gripper widths of the same episode.

**Why a separate Python 3.9 environment?** The zarr file has to be written with the library versions UMI itself uses (`zarr 2.16`, `numcodecs 0.11`, `imagecodecs 2023.9.18`). Colab's default Python is too new for those. The environment here is small (five packages, about 1-2 minutes), because converting needs none of PyTorch.

**One setting to remember for Phase 5:** UMI's real data is 60 Hz and its config subsamples it by 3. Your sim demos are 10 Hz, so training must use `task.obs_down_sample_steps=1` (the config file itself marks it `# 3, 1`).

## 1. Paths and Google Drive

In [ ]:
USE_DRIVE = True

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/umi_mujoco'
else:
    ROOT = '/content/umi_mujoco'

DEMO_DIR = f'{ROOT}/sim_demos'
DATA_DIR = f'{ROOT}/data'
os.makedirs(DATA_DIR, exist_ok=True)
LOCAL_ZARR = '/content/sim_cup.zarr.zip'          # written locally first (zip files are slow to write on Drive)
REPO = '/content/universal_manipulation_interface'
ENV = '/content/umi_data_env'
PYD = f'{ENV}/bin/python'
os.environ.update(ROOT=ROOT, DEMO_DIR=DEMO_DIR, DATA_DIR=DATA_DIR, LOCAL_ZARR=LOCAL_ZARR, REPO=REPO, ENV=ENV, PYD=PYD)

import glob
n = len(glob.glob(f'{DEMO_DIR}/ep_*.npz'))
print(n, 'demos found in', DEMO_DIR)
assert n > 0, 'no demos found - run Phase 3 first'

## 2. Create the small Python 3.9 environment (about 1-2 minutes)

In [ ]:
%%bash
set -e
pip -q install uv
uv python install 3.9
uv venv "$ENV" --python 3.9 --seed
uv pip install --python "$PYD" numpy==1.24.4 scipy==1.11.4 zarr==2.16.1 numcodecs==0.11.0 imagecodecs==2023.9.18
"$PYD" - <<'EOF'
import numpy, scipy, zarr, numcodecs, imagecodecs, sys
print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| zarr', zarr.__version__,
      '| numcodecs', numcodecs.__version__, '| imagecodecs', imagecodecs.__version__)
print('JPEG-XL available:', imagecodecs.JPEGXL.available)
EOF

Expected: `zarr 2.16.1`, `numcodecs 0.11.0`, `imagecodecs 2023.9.18` and `JPEG-XL available: True`. If the last one says `False`, paste the output to me.

## 3. Get the UMI repository
The converter uses UMI's own `ReplayBuffer` and JPEG-XL codec classes, so the file is written the same way as by the original script.

In [ ]:
%%bash
if [ ! -d "$REPO/.git" ]; then
  git clone --depth 1 https://github.com/real-stanford/universal_manipulation_interface.git "$REPO"
fi
cd "$REPO" && git log -1 --format='repo at %h (%ad)'
ls "$REPO/diffusion_policy/common/replay_buffer.py" "$REPO/diffusion_policy/codecs/imagecodecs_numcodecs.py"

## 4. Write the converter and the verifier
- `convert_to_umi_zarr.py` - reads every `ep_*.npz`, writes the zarr file. It also stores the rotation vectors in a continuous form: with the gripper pointing straight down, every orientation sits right at the angle where a rotation vector has two equivalent spellings, and the raw values would jump between them from frame to frame. The rotations are identical either way.
- `verify_umi_zarr.py` - re-opens the file the way `UmiDataset` does and checks it (step 6).

In [ ]:
%%writefile /content/convert_to_umi_zarr.py
"""Convert simulated demos (ep_*.npz from Phase 3) into UMI's replay-buffer zarr.

Mirrors scripts_slam_pipeline/07_generate_replay_buffer.py of the UMI repo: same keys, dtypes, chunking and image
compression. It uses the repo's own ReplayBuffer and JpegXl classes, so the result is read by UmiDataset unchanged.

Run in the Python 3.9 data environment:
    python convert_to_umi_zarr.py --repo /content/universal_manipulation_interface \
        --demos /content/drive/MyDrive/umi_mujoco/sim_demos --output /content/sim_cup.zarr.zip
"""
import argparse, concurrent.futures, glob, json, multiprocessing, os, sys, time
import numpy as np
from scipy.spatial.transform import Rotation as R

MIN_FRAMES = 16     # action_horizon 16 with down-sampling 1 needs at least this many frames per episode


def unwrap_rotvec(rv):
    """A rotation by an angle phi near pi has two equivalent rotation vectors (n*phi and -n*(2*pi - phi)), and
    pointing the gripper straight down puts every orientation right at that point. Pick, frame by frame, the form
    closest to the previous frame so the stored trajectory is continuous. The rotations themselves do not change."""
    out = np.array(rv, dtype=np.float64)
    for t in range(1, len(out)):
        ang = np.linalg.norm(out[t])
        if ang < 1e-9:
            continue
        alt = out[t] - 2.0 * np.pi * out[t] / ang
        if np.linalg.norm(alt - out[t - 1]) < np.linalg.norm(out[t] - out[t - 1]):
            out[t] = alt
    return out


def load_lowdim_episode(path):
    """Low-dimensional arrays of one demo, in the layout 07_generate_replay_buffer.py writes."""
    with np.load(path) as z:
        pos = np.asarray(z["robot0_eef_pos"], dtype=np.float64)
        rot = unwrap_rotvec(np.asarray(z["robot0_eef_rot_axis_angle"], dtype=np.float64))
        width = np.asarray(z["robot0_gripper_width"], dtype=np.float64).reshape(-1)
        record_dt = float(z["record_dt"])
    pose = np.concatenate([pos, rot], axis=-1)              # (T, 6): position + axis-angle
    start = np.empty_like(pose)
    start[:] = pose[0]
    end = np.empty_like(pose)
    end[:] = pose[-1]
    data = {
        "robot0_eef_pos": pos.astype(np.float32),
        "robot0_eef_rot_axis_angle": rot.astype(np.float32),
        "robot0_gripper_width": np.expand_dims(width, axis=-1).astype(np.float32),
        "robot0_demo_start_pose": start,                    # pose at the start of the demo, repeated for every frame
        "robot0_demo_end_pose": end,
    }
    return data, record_dt


def write_images(replay_buffer, tasks, n_total, out_res, compressor, workers, log=print):
    """tasks: list of (npz_path, buffer_start, n_frames). One zarr chunk per frame, so threads never share a chunk."""
    shape = (n_total,) + tuple(out_res) + (3,)
    arr = replay_buffer.data.require_dataset(
        name="camera0_rgb", shape=shape, chunks=(1,) + tuple(out_res) + (3,),
        compressor=compressor, dtype=np.uint8)

    def work(task):
        path, start, n = task
        with np.load(path) as z:
            imgs = z["camera0_rgb"]
        if imgs.dtype != np.uint8 or imgs.shape != (n,) + tuple(out_res) + (3,):
            raise ValueError("%s: images are %s %s, expected %s uint8" % (
                os.path.basename(path), imgs.dtype, imgs.shape, (n,) + tuple(out_res) + (3,)))
        arr[start:start + n] = imgs
        return n

    t0, done = time.time(), 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(work, t) for t in tasks]
        for i, f in enumerate(concurrent.futures.as_completed(futures), 1):
            done += f.result()
            if i % 10 == 0 or i == len(tasks):
                log("  images: %d/%d episodes, %d/%d frames, %.0f s" % (i, len(tasks), done, n_total, time.time() - t0))
    return arr


def build_lowdim(replay_buffer, files, log=print):
    """Add every usable episode to the buffer. Returns (tasks, names, record_dt)."""
    tasks, names, dts, start = [], [], set(), 0
    for path in files:
        data, record_dt = load_lowdim_episode(path)
        n = len(data["robot0_eef_pos"])
        if n < MIN_FRAMES:
            log("  skipping %s: only %d frames (need at least %d)" % (os.path.basename(path), n, MIN_FRAMES))
            continue
        replay_buffer.add_episode(data=data, compressors=None)
        tasks.append((path, start, n))
        names.append(os.path.basename(path))
        dts.add(round(record_dt, 6))
        start += n
    if len(dts) != 1:
        raise ValueError("episodes were recorded at different rates: %s" % sorted(dts))
    return tasks, names, dts.pop()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True, help="path of the cloned UMI repository")
    ap.add_argument("--demos", required=True, help="folder with ep_*.npz from Phase 3")
    ap.add_argument("--output", required=True, help="output path, e.g. sim_cup.zarr.zip")
    ap.add_argument("--max_episodes", type=int, default=None)
    ap.add_argument("--compression_level", type=int, default=99, help="JPEG-XL level, same default as UMI")
    ap.add_argument("--workers", type=int, default=None)
    args = ap.parse_args()

    sys.path.insert(0, args.repo)
    import zarr
    from diffusion_policy.common.replay_buffer import ReplayBuffer
    from diffusion_policy.codecs.imagecodecs_numcodecs import register_codecs, JpegXl
    register_codecs()

    files = sorted(glob.glob(os.path.join(args.demos, "ep_*.npz")))
    if args.max_episodes:
        files = files[:args.max_episodes]
    if not files:
        raise SystemExit("no ep_*.npz files in " + args.demos)
    print("%d demo files found" % len(files))

    out_res = (224, 224)
    rb = ReplayBuffer.create_empty_zarr(storage=zarr.MemoryStore())
    tasks, names, record_dt = build_lowdim(rb, files)
    n_total = sum(t[2] for t in tasks)
    print("%d episodes, %d frames, %.1f Hz" % (len(tasks), n_total, 1.0 / record_dt))

    workers = args.workers or min(4, multiprocessing.cpu_count())
    compressor = JpegXl(level=args.compression_level, numthreads=1)
    write_images(rb, tasks, n_total, out_res, compressor, workers)

    if os.path.exists(args.output):
        os.remove(args.output)
    print("saving", args.output)
    with zarr.ZipStore(args.output, mode="w") as zip_store:
        rb.save_to_store(store=zip_store)

    info = {
        "n_episodes": len(tasks), "n_frames": n_total, "record_dt": record_dt, "record_hz": 1.0 / record_dt,
        "image_size": list(out_res), "episodes": names,
        "training_overrides": ["task.obs_down_sample_steps=1", "task.dataset_frequeny=0"],
    }
    info_path = args.output.replace(".zarr.zip", "") + ".info.json"
    with open(info_path, "w") as f:
        json.dump(info, f, indent=1)
    print("done: %.1f MB, info in %s" % (os.path.getsize(args.output) / 1e6, info_path))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/verify_umi_zarr.py
"""Check that a converted zarr is what UMI's training code expects. Run in the Python 3.9 data environment:
    python verify_umi_zarr.py --repo REPO --zarr sim_cup.zarr.zip [--demos DEMO_DIR] [--compare cup_in_the_wild.zarr.zip]
"""
import argparse, json, os, sys
import numpy as np

REQUIRED = ("camera0_rgb", "robot0_eef_pos", "robot0_eef_rot_axis_angle", "robot0_gripper_width",
            "robot0_demo_start_pose", "robot0_demo_end_pose")


def sim_shape_meta(ds=1, obs_horizon=2, action_horizon=16, img=224):
    """What diffusion_policy/config/task/umi.yaml resolves to, with the sim settings (10 Hz data -> down-sampling 1)."""
    def low(shape, **kw):
        d = {"shape": shape, "horizon": obs_horizon, "latency_steps": 0, "down_sample_steps": ds, "type": "low_dim"}
        d.update(kw)
        return d
    return {
        "obs": {
            "camera0_rgb": {"shape": [3, img, img], "horizon": obs_horizon, "latency_steps": 0,
                            "down_sample_steps": ds, "type": "rgb"},
            "robot0_eef_pos": low([3]),
            "robot0_eef_rot_axis_angle": low([6], raw_shape=[3], rotation_rep="rotation_6d"),
            "robot0_gripper_width": low([1]),
            "robot0_eef_rot_axis_angle_wrt_start": low([6], raw_shape=[3]),
        },
        "action": {"shape": [10], "horizon": action_horizon, "latency_steps": 0, "down_sample_steps": ds,
                   "rotation_rep": "rotation_6d"},
    }


def make_sampler(rb, shape_meta):
    """Build the sampler exactly as UmiDataset.__init__ does."""
    from diffusion_policy.common.sampler import SequenceSampler
    rgb_keys, lowdim_keys, key_horizon, key_latency, key_ds = [], [], {}, {}, {}
    for key, attr in shape_meta["obs"].items():
        (rgb_keys if attr.get("type") == "rgb" else lowdim_keys).append(key)
        key_horizon[key], key_latency[key], key_ds[key] = attr["horizon"], attr["latency_steps"], attr["down_sample_steps"]
    a = shape_meta["action"]
    key_horizon["action"], key_latency["action"], key_ds["action"] = a["horizon"], a["latency_steps"], a["down_sample_steps"]
    sampler_lowdim = [k for k in lowdim_keys if "wrt" not in k]
    for key in rb.keys():
        if key.endswith("_demo_start_pose") or key.endswith("_demo_end_pose"):
            sampler_lowdim.append(key)
            q = key.split("_")[0] + "_eef_pos"
            key_horizon[key] = shape_meta["obs"][q]["horizon"]
            key_latency[key] = shape_meta["obs"][q]["latency_steps"]
            key_ds[key] = shape_meta["obs"][q]["down_sample_steps"]
    sampler = SequenceSampler(
        shape_meta=shape_meta, replay_buffer=rb, rgb_keys=rgb_keys, lowdim_keys=sampler_lowdim,
        key_horizon=key_horizon, key_latency_steps=key_latency, key_down_sample_steps=key_ds,
        episode_mask=None, action_padding=False, repeat_frame_prob=0.0, max_duration=None)
    return sampler


def check_structure(rb):
    results = []
    keys = list(rb.keys())
    results.append(("required keys present", all(k in keys for k in REQUIRED),
                    "missing: %s" % [k for k in REQUIRED if k not in keys] if not all(k in keys for k in REQUIRED) else "all %d" % len(REQUIRED)))
    ends = np.asarray(rb.episode_ends[:])
    n = int(ends[-1]) if len(ends) else 0
    results.append(("episode_ends increasing, last = total frames", len(ends) > 0 and bool(np.all(np.diff(ends) > 0)), "%d episodes, %d frames" % (len(ends), n)))
    shapes_ok = all(rb[k].shape[0] == n for k in keys)
    results.append(("every array has one row per frame", shapes_ok, ""))
    exp = {"robot0_eef_pos": ((3,), "float32"), "robot0_eef_rot_axis_angle": ((3,), "float32"),
           "robot0_gripper_width": ((1,), "float32"), "robot0_demo_start_pose": ((6,), None),
           "robot0_demo_end_pose": ((6,), None), "camera0_rgb": ((224, 224, 3), "uint8")}
    bad = []
    for k, (shape, dt) in exp.items():
        if k in keys:
            arr = rb[k]
            if tuple(arr.shape[1:]) != shape or (dt is not None and str(arr.dtype) != dt):
                bad.append("%s is %s %s" % (k, arr.shape, arr.dtype))
    results.append(("shapes and dtypes as in the real dataset", not bad, "; ".join(bad)))
    lengths = np.diff(np.concatenate([[0], ends]))
    results.append(("every episode long enough for 16 actions", bool(lengths.min() >= 16), "shortest %d frames" % lengths.min()))
    finite = all(np.isfinite(np.asarray(rb[k][:])).all() for k in keys if k != "camera0_rgb")
    results.append(("no NaN / inf in the low-dim data", bool(finite), ""))
    return results


def sampler_dry_run(rb, ds=1, n_samples=25, seed=0):
    """Draw sequences the way the training dataloader does and push them through UMI's own relative-pose code."""
    from umi.common.pose_util import pose_to_mat, mat_to_pose10d
    from diffusion_policy.common.pose_repr_util import convert_pose_mat_rep
    meta = sim_shape_meta(ds=ds)
    sampler = make_sampler(rb, meta)
    results = [("sampler builds, usable training samples", len(sampler) > 0, "%d samples" % len(sampler))]
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(sampler), size=min(n_samples, len(sampler)), replace=False)
    shapes_ok, ident_err, rel_pos, grip, finite, extra = True, 0.0, [], [], True, ""
    ident6 = mat_to_pose10d(np.eye(4))[3:]
    for i in idxs:
        s = sampler.sample_sequence(int(i))
        want = {"camera0_rgb": (2, 224, 224, 3), "robot0_eef_pos": (2, 3), "robot0_eef_rot_axis_angle": (2, 3),
                "robot0_gripper_width": (2, 1), "robot0_demo_start_pose": (2, 6), "action": (16, 7)}
        for k, shp in want.items():
            if k not in s or tuple(s[k].shape) != shp:
                shapes_ok = False
                extra = "%s: got %s, expected %s" % (k, None if k not in s else tuple(s[k].shape), shp)
        pose = pose_to_mat(np.concatenate([s["robot0_eef_pos"], s["robot0_eef_rot_axis_angle"]], axis=-1).astype(np.float64))
        act = pose_to_mat(s["action"][..., :6].astype(np.float64))
        act_rel = mat_to_pose10d(convert_pose_mat_rep(act, base_pose_mat=pose[-1], pose_rep="relative", backward=False))
        finite &= bool(np.isfinite(act_rel).all())
        ident_err = max(ident_err, float(np.abs(act_rel[0, :3]).max()), float(np.abs(act_rel[0, 3:] - ident6).max()))
        rel_pos.append(np.abs(act_rel[:, :3]).max())
        grip.append((s["action"][:, 6].min(), s["action"][:, 6].max()))
    results.append(("sequences have the shapes the model expects", shapes_ok, extra or "obs (2,...), action (16,7)"))
    results.append(("first action = current pose (relative action is identity)", ident_err < 1e-4, "max deviation %.1e" % ident_err))
    results.append(("relative actions are finite", finite, ""))
    rp = np.array(rel_pos)
    results.append(("relative action sizes are plausible", float(rp.max()) < 1.0,
                    "largest displacement within 1.6 s: median %.2f m, max %.2f m" % (np.median(rp), rp.max())))
    return results


def describe(path, title):
    import zarr
    print("--- %s: %s" % (title, path))
    with zarr.ZipStore(path, mode="r") as zs:
        root = zarr.group(store=zs)
        meta_ends = root["meta"]["episode_ends"][:]
        print("   episodes: %d   frames: %d" % (len(meta_ends), int(meta_ends[-1])))
        out = {}
        for k, arr in root["data"].arrays():
            comp = arr.compressor.codec_id if arr.compressor is not None else "none"
            print("   %-28s shape=%-20s dtype=%-8s chunks=%-18s compressor=%s" % (k, arr.shape, arr.dtype, arr.chunks, comp))
            out[k] = (arr.shape[1:], str(arr.dtype), arr.chunks[1:], comp)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--zarr", required=True)
    ap.add_argument("--demos", default=None, help="folder with the source ep_*.npz (enables the image round-trip check)")
    ap.add_argument("--compare", default=None, help="path of the real cup_in_the_wild.zarr.zip, to compare layouts")
    ap.add_argument("--save_frames", default=None, help="write decoded sample frames to this .npy")
    args = ap.parse_args()
    sys.path.insert(0, args.repo)
    import zarr
    from diffusion_policy.common.replay_buffer import ReplayBuffer
    from diffusion_policy.codecs.imagecodecs_numcodecs import register_codecs
    register_codecs()

    ours = describe(args.zarr, "converted sim dataset")
    if args.compare and os.path.exists(args.compare):
        real = describe(args.compare, "real UMI dataset")
        print("\n--- layout comparison (per-frame shape, dtype, chunks, compressor)")
        for k in sorted(set(ours) | set(real)):
            if k in ours and k in real:
                print("   %-28s %s" % (k, "identical" if ours[k] == real[k] else "differs: ours %s | real %s" % (ours[k], real[k])))
            else:
                print("   %-28s only in the %s dataset" % (k, "sim" if k in ours else "real"))
    print()

    # load exactly like UmiDataset does
    with zarr.ZipStore(args.zarr, mode="r") as zs:
        rb = ReplayBuffer.copy_from_store(src_store=zs, store=zarr.MemoryStore())

    results = check_structure(rb)
    results += sampler_dry_run(rb)

    if args.demos:
        info_path = args.zarr.replace(".zarr.zip", "") + ".info.json"
        if os.path.exists(info_path):
            info = json.load(open(info_path))
            ends = np.asarray(rb.episode_ends[:])
            starts = np.concatenate([[0], ends[:-1]])
            rng = np.random.default_rng(1)
            errs, frames = [], []
            for e in rng.choice(len(ends), size=min(5, len(ends)), replace=False):
                local = int(rng.integers(0, ends[e] - starts[e]))
                with np.load(os.path.join(args.demos, info["episodes"][e])) as z:
                    src = z["camera0_rgb"][local].astype(np.float32)
                dec = rb["camera0_rgb"][int(starts[e]) + local]
                errs.append(float(np.abs(dec.astype(np.float32) - src).mean()))
                frames.append(np.stack([src.astype(np.uint8), dec]))
            results.append(("decoded images match the originals", max(errs) < 5.0,
                            "mean abs pixel error %s (0-255 scale)" % ", ".join("%.2f" % x for x in errs)))
            if args.save_frames:
                np.save(args.save_frames, np.stack(frames))

    n_fail = 0
    print("CHECKS")
    for name, ok, msg in results:
        n_fail += (not ok)
        print("  [%s] %-58s %s" % ("PASS" if ok else "FAIL", name, msg))
    print("\n%d/%d checks passed" % (len(results) - n_fail, len(results)))
    sys.exit(1 if n_fail else 0)


if __name__ == "__main__":
    main()

## 5. Convert
Reads the demos from Drive and writes `/content/sim_cup.zarr.zip`, then copies it (and a small `.info.json`) to Drive. For 200 demos expect roughly 5-15 minutes, mostly JPEG-XL compression of about 34,000 frames.

In [ ]:
%%bash
set -e
"$PYD" /content/convert_to_umi_zarr.py --repo "$REPO" --demos "$DEMO_DIR" --output "$LOCAL_ZARR"
cp "$LOCAL_ZARR" "$DATA_DIR/"
cp "${LOCAL_ZARR%.zarr.zip}.info.json" "$DATA_DIR/"
ls -lh "$DATA_DIR"

## 6. Verify
Re-opens the file exactly like `UmiDataset` does, then draws training sequences with UMI's own `SequenceSampler` and runs them through UMI's relative-pose code (the same maths as the training dataloader, only without PyTorch). It checks that:
- all required keys, shapes and dtypes are there,
- the sequences have the shapes the model expects (2 observation frames, 16 actions),
- the first action equals the current pose (relative action = identity), which catches pose-convention mistakes,
- decoded images match the originals.

If `cup_in_the_wild.zarr.zip` from Phase 0 is on Drive, its layout is printed and compared with yours. **Every key present in both should read `identical`.** `robot0_demo_*` keys may differ in dtype only if you see a note saying so - tell me if anything else differs.

In [ ]:
%%bash
COMPARE=""
if [ -f "$DATA_DIR/cup_in_the_wild.zarr.zip" ]; then COMPARE="--compare $DATA_DIR/cup_in_the_wild.zarr.zip"; fi
"$PYD" /content/verify_umi_zarr.py --repo "$REPO" --zarr "$LOCAL_ZARR" --demos "$DEMO_DIR" \
    --save_frames /content/decoded_check.npy $COMPARE

## 7. Look at decoded frames
Top row: the original simulator image. Bottom row: the same frame after being compressed into the zarr file and decoded again. They should look the same.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
pairs = np.load('/content/decoded_check.npy')            # (n, 2, 224, 224, 3)
n = len(pairs)
fig, axes = plt.subplots(2, n, figsize=(3 * n, 6.2))
for j in range(n):
    for i, label in enumerate(['original', 'from zarr']):
        axes[i, j].imshow(pairs[j, i]); axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(label, loc='left')
plt.show()

## Phase 4 checklist

- [ ] Step 2 shows `JPEG-XL available: True`
- [ ] Step 5 finished and `sim_cup.zarr.zip` is on Drive in `umi_mujoco/data`
- [ ] Step 6: every check reads `PASS`
- [ ] Step 7: original and decoded frames look the same

**Keep for Phase 5 (training):** the dataset path `umi_mujoco/data/sim_cup.zarr.zip`, and the override `task.obs_down_sample_steps=1`.

Next: Phase 5 - train UMI's diffusion policy on this dataset. The default model (CLIP ViT-B/16, batch size 64) is meant for a large GPU, so the notebook will use a lighter vision backbone and a smaller batch for Colab.